In [ ]:
using Plots, DifferentialEquations, LaTeXStrings
include("imode_sigmoid.jl")
using .ImodeSigmoid

In [ ]:
function Network!(du,u,p,t)

	neuron_currents = u[1:4:n_neurons*4] .- Idc/2
	neuron_currents[neuron_currents .< 0] .= 0

	for i = 1:n_neurons
		If, Is, Ius, Isyn = u[4*i-3:4*i]

		sigf_pars = (Ithr=Ithr_f, Igain=Igain_f-Is, Ilin=Ilin_f)
		sigs_pars = (Ithr=Ithr_s, Igain=Igain_s-Ius, Ilin=Ilin_s)

		du[4*i-3] = (-If - Is - Ius + Imode_sigmoid_eval(If, sigf_pars, var_gain=true) + Imode_sigmoid_eval(Is, sigs_pars, var_gain=true) + Isyn) / τ_f
		du[4*i-2] = (-Is + If) / τ_s
		du[4*i-1] = (-Ius + If) / τ_us

		# Synapse
		du[4*i] = (-Isyn + sum(A[:,i].*neuron_currents) + Idc) / τ_us
	end
end

In [ ]:
A = [0 -1 -1 0; -1 0 0 -1; -1 0 0 -1; 0 -1 -1 0]
n_neurons = size(A)[1]

In [ ]:
τ_f = 0.0001
τ_s = 0.02
τ_us = 0.4

Ithr_f = 150e-9
Igain_f = 700e-9
Ilin_f = 250e-9

Ithr_s = 110e-9
Igain_s = 250e-9
Ilin_s = 50e-9

Idc = 400e-9

In [ ]:
Tfinal = 3.
tspan = (0.0, Tfinal)

x0 = zeros(n_neurons*4) .+ 1e-9
x0[4] = 1e-8 # Perturbation to avoid symmetry
x0[12] = 1e-8 # Perturbation to avoid symmetry

pars = (Ithr_f, Igain_f, Ilin_f, Ithr_s, Igain_s, Ilin_s)

prob = ODEProblem(Network!, x0, tspan, pars)
sol = solve(prob, Rodas4P(), abstol=1e-15, reltol=1e-12);

In [ ]:
plot(sol, idxs=[1, 5], layout=(2, 1), subplot=1, xlabel="Time (s)", ylabel="Current (A)")
plot!(sol, idxs=[9, 13], layout=(2, 1), subplot=2, xlabel="Time (s)", ylabel="Current (A)", color=[3 4])